In [0]:
%sql
drop table if exists workspace.gold.dim_date;


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_date (
  id_date BIGINT,
  date DATE,
  year INT,
  quarter INT,
  month INT,
  month_name STRING,
  day INT,
  day_name STRING,
  week_of_year INT,
  is_weekend BOOLEAN

)
""")

In [0]:
from pyspark.sql import functions as F
df_spark = spark.table("workspace.silver.tbl_ventas_detalle")
dim=(df_spark
    .select(F.col("fecha").alias("date"))
    .filter(F.col("date").isNotNull())
    .dropDuplicates(["date"])
    .orderBy("date")
)

In [0]:
dim = (
    dim
    .withColumn( "id_date",F.date_format("date", "yyyyMMdd").cast("bigint"))
    .withColumn( "year",F.year("date"))
    .withColumn( "month",F.month("date"))
    .withColumn( "day", F.dayofmonth("date"))
    .withColumn( "day_name",F.date_format("date", "EEEE"))
    .withColumn( "week_of_year",F.weekofyear("date"))
    .withColumn( "month_name",F.date_format("date", "MMMM"))
    .withColumn( "quarter",F.quarter("date"))
    .withColumn( "is_weekend",F.dayofweek("date").isin([1, 7]))
)

In [0]:
dim.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_date")


In [0]:
%sql
select *from workspace.gold.dim_date